# 005 Build robustness, validation, and divergence analyses

## Objective

Build the classification sensitivities, validation samples, candidate-divergence estimates, uncertainty intervals, and lexical sensitivities required for the PNAS Nexus Research Report. The run succeeds only against the canonical data contract and excludes records after 2025.


## Setup and reproducibility

The analysis uses a fixed seed and 2,000 bootstrap replicates. Set `FETCH_ARXIV = True` only when network access is available and the primary-category validation CSV has not yet been completed.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

def find_workspace(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'config' / 'workspace.json').exists() or (candidate / '.zenodo.json').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the reproduction bundle')

WORKSPACE_ROOT = find_workspace(Path.cwd().resolve())
SCRIPTS_DIR = WORKSPACE_ROOT / 'code'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from robustness_analysis import END_YEAR, N_BOOTSTRAP, SEED, build_all

FETCH_ARXIV = False
print({'workspace': str(WORKSPACE_ROOT), 'end_year': END_YEAR, 'seed': SEED, 'bootstrap': N_BOOTSTRAP})


## Analysis plan

1. Validate the canonical input hashes and write the analysis manifest.
2. Compare primary, any-class, and fractional field assignments and measure cross-listing.
3. Prepare blinded candidate-dictionary annotations and score them when complete.
4. Bootstrap annual candidate divergence and the prespecified early-versus-late contrast.
5. Measure lexical divergence with and without candidate terms.


In [ ]:
result = build_all(
    project_root=WORKSPACE_ROOT,
    n_bootstrap=N_BOOTSTRAP,
    fetch_arxiv=FETCH_ARXIV,
)
result['summary']


## Results and acceptance gates

The summary permits `increasing divergence` only when the late-minus-early interval excludes zero in the positive direction. Candidate-dictionary release readiness remains false until blinded holdout annotations are complete and both precision and recall pass 0.90.


In [ ]:
display(result['candidate']['candidate_divergence_contrast'])
display(result['validation'])
display(result['classification']['classification_crosslisting_yearly'].tail())


## Next steps

- Complete the development annotation sheet before changing the dictionary.
- Freeze the dictionary, then complete the two holdout sheets without consulting the system key.
- Re-run this notebook and use only the generated tables in manuscript figures and claims.
